# VAT Identifier Discovery, UK Company VAT Numbers

# Part 1: Research

## 1.1 What a VAT number actually is

A UK VAT number is 9 digits. The last 2 are a check, worked out from the first 7 using one of two HMRC rules (older registrations use "mod 97", newer ones use "mod 97 + 55"). A few rarer formats exist (12-digit branch numbers, `GD` for government, `HA` for health bodies), but they're a small slice, so this project only targets the normal 9-digit form.

Why the check digits matter: they let us throw out fake numbers for free, before spending an HMRC call on them.

One more number worth knowing before anything else: about 4.2 million companies are live in the UK, but only about 2.18 million VAT registrations exist nationally (and that count includes sole traders, not just companies). So most companies simply don't have a VAT number.

In [ ]:

import random

WEIGHTS = (8, 7, 6, 5, 4, 3, 2)

def _residual(total: int) -> int:
    while total > 0:
        total -= 97
    return abs(total)

def vrn_checksum_valid(vrn: str) -> bool:
    digits = "".join(ch for ch in vrn if ch.isdigit())
    if len(digits) != 9:
        return False
    total = sum(int(d) * w for d, w in zip(digits[:7], WEIGHTS))
    check = int(digits[7:9])
    return _residual(total) == check or _residual(total + 55) == check

def make_valid(prefix7: int, scheme: str = "97") -> str:
    p = f"{prefix7:07d}"
    total = sum(int(d) * w for d, w in zip(p, WEIGHTS))
    if scheme == "9755":
        total += 55
    return p + f"{_residual(total):02d}"

random.seed(0)

# round trip check
ok = all(vrn_checksum_valid(make_valid(random.randrange(10**7), s))
         for _ in range(2000) for s in ("97", "9755"))
print("round-trip ok:", ok)

# how much of the space is valid?
n = 200_000
hits = sum(vrn_checksum_valid(f"{random.randrange(10**9):09d}") for _ in range(n))
print(f"random 9-digit strings that pass: {hits:,}/{n:,} = {hits/n:.2%} (math says 2.00%)")

# does a typo always fail?
v = make_valid(1234567, "97")
survive = sum(vrn_checksum_valid(v[:i] + d + v[i+1:])
              for i in range(9) for d in "0123456789" if d != v[i])
print(f"example valid number {v}: {survive} of 81 single-digit typos still pass")

round-trip ok: True


random 9-digit strings that pass: 3,916/200,000 = 1.96% (math says 2.00%)
example valid number 123456782: 1 of 81 single-digit typos still pass


Reading this: the checksum kills about 98% of random noise for free. But 2% of pure garbage still gets through, and because two rules exist, about 1 in 80 single-digit typos of a real number *also* passes. So a checksum pass is a reason to check with HMRC, never a reason to ship a number as fact.

## 1.2 The checker only runs backwards

You give HMRC's checker a VAT number, it tells you if it's real and whose it is. There's no way to search the other direction: no box where you type a company name and get a number back.

Two things follow from that:
1. Something else has to generate the guesses. HMRC only confirms or kills them.
2. HMRC's registered name for a company often doesn't match Companies House or a website ("Ltd" vs "Limited", trading name vs legal name). So even a correct guess still needs a fuzzy name check at the end, and that's exactly where a wrong match can sneak through.

In [ ]:

import json, urllib.request, urllib.error

def hmrc_check(vrn: str, timeout: int = 15):
    url = f"https://api.service.hmrc.gov.uk/organisations/vat/check-vat-number/lookup/{vrn}"
    req = urllib.request.Request(url, headers={"Accept": "application/vnd.hmrc.2.0+json"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode()[:300]
    except Exception as e:
        return None, f"no network from here ({type(e).__name__})"

probe = (make_valid(9998887, "97"))
print("probe VRN:", probe)
print(hmrc_check(probe))

probe VRN: 999888789


(403, 'Host not in allowlist: api.service.hmrc.gov.uk. Add this host to your network egress settings to allow access.')


## 1.3 Where a number could actually be found

| # | Source | Good | Bad |
|---|---|---|---|
| 1 | Company's own website | Clean, in the company's own words | UK law only forces this for sites that actually sell online (2002 Ecommerce Regs). A brochure site has no such duty |
| 2 | Invoices from real transactions | Legally required, most reliable | Only works if you've actually bought something. The customer here has this for all 40k suppliers already, so it's step zero |
| 3 | Asking the company directly | Accurate if they answer | Doesn't scale to thousands, slow, patchy reply rate |
| 4 | Third-party lookup sites | Someone already did the search | Partial coverage, no idea where their data comes from, rate-limited (tested below) |
| 5 | Public procurement records | Suppliers state it as a required field on some forms | Only covers companies that sold to government, and often the wrong party's number is on the document (tested in Part 2) |
| 6 | Council spend data | Some councils publish a VAT column | No fixed format, differs council to council |
| 7 | Bulk web crawl (Common Crawl) | Free, no per-site limits, scales | Coverage is uneven, small sites may be missing or old |
| 8 | EORI (adjacent ID) | Exact math, not a guess (see 1.8) | Only importers/exporters have one |
| 9 | VIES (EU checker, for Beyond-the-UK) | Free, no signup | Doesn't cover the UK post-Brexit |

Sources 1 and 7 are the ones that actually scale. 2 is free and already sitting with the customer. The rest are patch-up tools for specific gaps.

## 1.4 Companies House: the list, not the answer

Companies House gives a free monthly file of every live UK company: name, number, address, SIC code, incorporation date, filing category. **No website field, no VAT field.** It's the list of who to check, not part of the answer.

What it does buy us: a way to skip effort before crawling anything. A company filed as "dormant" or with no accounts on file is very unlikely to have a live website worth visiting, so it can go to the back of the queue. A trading e-commerce company is much more likely to be legally required to show a VAT number (rule 1 above), so it should go to the front.

In [ ]:

import csv, io, glob, random, zipfile
random.seed(1)

SNAPSHOT = next(iter(glob.glob("BasicCompanyData-2026-08-01-part1_7.zip")), None)

def load_rows():
    if SNAPSHOT:
        with zipfile.ZipFile(SNAPSHOT) as z, z.open(z.namelist()[0]) as f:
            rdr = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8"))
            for row in rdr:
                yield {"name": row["CompanyName"].strip(), "number": row[" CompanyNumber"].strip(),
                       "status": row["CompanyStatus"].strip(), "sic": row["SICCode.SicText_1"].strip(),
                       "accounts": row["Accounts.AccountCategory"].strip()}
    else:
        sics = ["47910 - retail via internet", "62012 - business software",
                "64209 - holding companies", "68209 - letting of real estate"]
        accts = ["DORMANT", "MICRO ENTITY", "SMALL", "FULL", "NO ACCOUNTS FILED"]
        for i in range(5000):
            yield {"name": f"SYNTH CO {i:05d} LIMITED", "number": f"{random.randrange(10**8):08d}",
                   "status": random.choice(["Active"]*9 + ["Liquidation"]),
                   "sic": random.choice(sics), "accounts": random.choices(accts, weights=[25,25,20,10,3])[0]}

def priority(row):
    s = 0
    if row["status"] != "Active": s -= 100
    if row["accounts"] in ("DORMANT", "NO ACCOUNTS FILED"): s -= 50
    if row["sic"].startswith(("6420", "6820")): s -= 20
    if row["sic"].startswith("479"): s += 30
    if row["accounts"] in ("SMALL", "FULL"): s += 10
    return s

rows = list(load_rows())
n_skip = sum(1 for r in rows if priority(r) < 0)
print(f"frame: {len(rows):,} companies ({'real snapshot' if SNAPSHOT else 'made-up demo'})")
print(f"pushed to back of queue (inactive/dormant/shell): {n_skip:,} = {n_skip/len(rows):.1%}")

frame: 5,000 companies (made-up demo)
pushed to back of queue (inactive/dormant/shell): 3,480 = 69.6%


## 1.5 Pulling a candidate off a page

The plan for any page of text: find a 9-digit shape → check it sits near the word "VAT" → check it passes the checksum → only *then* ask HMRC. Tested on 5 made-up pages, including 3 traps: a phone number, a company registration number, a price sitting next to "excl. VAT".

In [4]:
import re

VRN_RE = re.compile(r"(?ix)(?<!\d)(?:GB\s*)?(\d{3})[\s.\-]?(\d{4})[\s.\-]?(\d{2})(?!\d)")
CONTEXT_RE = re.compile(r"(?i)\bv\.?a\.?t\b")

def extract(html: str, window: int = 80):
    text = re.sub(r"<[^>]+>", " ", html)
    found = []
    for m in VRN_RE.finditer(text):
        nine = "".join(m.groups())
        ctx = " ".join(text[max(0, m.start()-window): m.end()+window].split())
        found.append((nine, bool(CONTEXT_RE.search(ctx))))
    return found

fixtures = {
  "footer, spaced + GB":  '<footer>J Smith Building Services Ltd. Reg 04523789. VAT Reg No: GB 123 4567 82</footer>',
  "terms page, unspaced": '<p>Our VAT registration number is 123456782 and our company number is 08881234.</p>',
  "trap: phone number":   '<div>Call us on 0161 4960 12. Established 1998.</div>',
  "trap: company ref":    '<span>Company No. 045237891 | Registered office: Leeds</span>',
  "trap: price near VAT": '<td>Invoice total 384736219 pence excl. VAT</td>',
}

print(f"{'fixture':24s} {'candidate':11s} {'ctx':4s} {'chksum':7s} verdict")
for name, html in fixtures.items():
    hits = extract(html) or [(None, None)]
    for nine, ctx in hits:
        if nine is None:
            print(f"{name:24s} {'-':11s} {'-':4s} {'-':7s} nothing found"); continue
        cs = vrn_checksum_valid(nine)
        verdict = "-> ask HMRC" if (ctx and cs) else "dropped"
        print(f"{name:24s} {nine:11s} {str(ctx):4s} {str(cs):7s} {verdict}")

fixture                  candidate   ctx  chksum  verdict
footer, spaced + GB      123456782   True True    -> ask HMRC
terms page, unspaced     123456782   True True    -> ask HMRC
trap: phone number       -           -    -       nothing found
trap: company ref        045237891   False False   dropped
trap: price near VAT     384736219   True False   dropped


Both real fixtures make it through. The phone number and company-number traps get caught, one by context, one by checksum. The nastiest trap, a price next to "excl. VAT", only dies at the checksum step, meaning at real scale, some fake candidates like this *will* slip through and only HMRC catches them. That's the whole reason step 4 is always HMRC, never a guess shipped as fact.

## 1.6 Trying it at scale: Common Crawl

Visiting 40,000 company sites one at a time from a laptop is slow and gets you blocked fast. Common Crawl already crawled a big slice of the web and lets you ask "what pages exist for this domain" without crawling it yourself. The plan: resolve company → domain, ask Common Crawl which pages it has for that domain (terms, contact, homepage), pull only those, run them through the funnel above.



## 1.7 Dead ends found along the way

- **A different third-party VAT lookup site.** Blocked my IP after about 50 requests while I tried to profile its coverage. Confirms these sites are built for the occasional human lookup, not a pipeline.
- **No forward register exists, by law.** Freedom-of-Information requests asking HMRC for a name-to-VAT list get refused: individual taxpayer data is protected under the Commissioners for Revenue and Customs Act 2005, s.18. This is exactly why nobody sells a complete list, the real one legally can't be published.
- **HMRC's bulk API history.** The old open version was shut down 17 Feb 2025. The new one needs a registered app, roughly a 2-week wait for approval.
- **Peppol** (EU e-invoicing network). Does carry VAT numbers for connected companies, but throttled to 2 queries a second and only covers companies already on the network. Fine as a backup for a small known list, not a main source.

## 1.8 EORI: a narrow shortcut

UK traders moving goods across the border get an EORI number, and since Brexit it's built directly from the VAT number:

```
EORI = "GB" + <9-digit VAT number> + "000"
```

So finding a company's EORI anywhere (customs paperwork, shipping databases) hands you its VAT number by construction, no guessing, no fuzzy matching. The catch: only companies that import or export have one at all. A local plumber never will, so this is high-precision but low-coverage.

## 1.9 What Part 1 concluded

A UK VAT dataset can be *partly* built from the open web. Full coverage isn't possible (no forward register exists, section 1.7), and most companies have no number to find at all (section 1.1). The honest product is "number where findable, with a measured error rate", not "the missing two thirds".

```
Companies House list (who to check, what to skip)
  -> candidates, best sources first: invoices already held -> procurement/council
     records -> company websites (Common Crawl at scale) -> EORI if found
  -> filter: shape -> context -> checksum
  -> HMRC check + name match
  -> record shipped with source, evidence, and date
```

Measured in this notebook: checksum pass rate on noise (~2%), typo survival rate (~1 in 80), how the funnel handles 5 test pages. From my own testing outside this notebook: the ~50-request IP block, Peppol's 2/sec cap. From documentation, still to double-check before quoting to a customer: the exact split of the 2.18M VAT registrations by business type, how many councils actually publish a VAT column, Common Crawl's real coverage of small UK sites.

---

# Part 2: Proof of concept

## 2.1 The sample

Picked real companies the honest way: Companies House's own advanced search, filtered to SIC code `62020` (IT consultancy), registered from Jan 2022 on, first page of results exactly as returned.

Picked IT consultancy on purpose, because Part 1 already guessed this sector would score badly: no online checkout, no legal reason to publish a VAT number. Testing on a sector I expected to lose in is a fairer test than picking one I expected to win in.

In [ ]:

sample_raw = [
    {"name": "MERAXUS ONLINE SOLUITION LTD", "number": "14686260", "status": "Dissolved"},
    {"name": "CYBER NEXUS LTD",               "number": "14686751", "status": "Dissolved"},
    {"name": "UNREAL SOLUTIONS LTD",          "number": "14451923", "status": "Active"},
    {"name": "DOX 4DEV LTD",                  "number": "14085415", "status": "Active"},
    {"name": "WINSYSTEM IT SERVICE COMPANY LTD", "number": "14757864", "status": "Active"},
    {"name": "JSM VENTURES LTD",              "number": "14853115", "status": "Active"},
    {"name": "BMTY GROUP LIMITED",            "number": "14104846", "status": "Active"},
    {"name": "HEALIO DIGITAL LTD",            "number": "14763873", "status": "Active"},
    {"name": "TECHKEE LIMITED",               "number": "15720228", "status": "Active"},
    {"name": "CYAN LTD",                      "number": "14841695", "status": "Active"},
    {"name": "FUTURIX IT CONSULTING LTD",     "number": "15719593", "status": "Active"},
    {"name": "MKAY CONSULTING LTD",           "number": "14866262", "status": "Active"},
    {"name": "TAZETTA SYSTEMS LIMITED",       "number": "14301748", "status": "Active"},
    {"name": "ELMAR HOME CARE LONDON LTD",    "number": "14545441", "status": "Active"},
    {"name": "COMPUTOLOGY LTD",               "number": "14316555", "status": "Active"},
    {"name": "JOHN CHADWICK CX LTD",          "number": "14754628", "status": "Active"},
    {"name": "SILICON MAFIA LIMITED",         "number": "14330075", "status": "Active"},
    {"name": "ARWOCK LTD",                    "number": "14690076", "status": "Dissolved"},
    {"name": "FARIO LIMITED",                 "number": "14691939", "status": "Dissolved"},
    {"name": "NOVUS SOFTWARE SOLUTIONS LTD",  "number": "14091065", "status": "Active"},
]

active = [r for r in sample_raw if r["status"] == "Active"]
print(f"20 companies returned, {len(active)} Active, {len(sample_raw)-len(active)} Dissolved (dropped, can't crawl a dead company's site)")
for r in active:
    print(" ", r["number"], r["name"])

20 companies returned, 16 Active, 4 Dissolved (dropped, can't crawl a dead company's site)
  14451923 UNREAL SOLUTIONS LTD
  14085415 DOX 4DEV LTD
  14757864 WINSYSTEM IT SERVICE COMPANY LTD
  14853115 JSM VENTURES LTD
  14104846 BMTY GROUP LIMITED
  14763873 HEALIO DIGITAL LTD
  15720228 TECHKEE LIMITED
  14841695 CYAN LTD
  15719593 FUTURIX IT CONSULTING LTD
  14866262 MKAY CONSULTING LTD
  14301748 TAZETTA SYSTEMS LIMITED
  14545441 ELMAR HOME CARE LONDON LTD
  14316555 COMPUTOLOGY LTD
  14754628 JOHN CHADWICK CX LTD
  14330075 SILICON MAFIA LIMITED
  14091065 NOVUS SOFTWARE SOLUTIONS LTD


## 2.2 Finding the websites

Searched for each of the 16 companies' real website. Kept "VAT" out of the search terms on purpose, to not fish for pages that already look promising.

**Result: 2 out of 16 (12.5%) had a findable, confirmable website.**

What killed the other 14: mostly no website at all. But also real cases of two unrelated companies sharing a near-identical name (Cyan, Techkee, Tazetta, Novus): none of the top search results actually matched the company number I was after, so I logged those as "no confident match" rather than guessing.

One more thing worth flagging: Elmar Home Care shows as "Active" at Companies House but a business-data site independently flags it as dormant. Active status doesn't mean the company is actually trading.

## 2.3 Running the funnel for real

Took the funnel from 1.5 and ran it on the two real pages that resolved.

In [ ]:

unreal_solutions_footer_and_body_excerpt = """
Runtime Video Recorder. The only video recording plugin for Unreal Engine
that works on desktop, mobile, and VR. Our customers: Amazon Games, Meta
Research. Subscription: $24.99 per month. Standard: $299 one-time.
Settings.VideoBitrate = 20000000; // 20 Mbps
Recorder->StartRecordingMultipleCameras(Cameras, "E:/Replays/multicam.mp4",
    30, 1920, 1080, Settings, true);
LinkedIn: https://www.linkedin.com/company/unreal-solutions-company/
(c) Unreal Solutions Ltd. Registered in England and Wales No. 14451923.
"""

dox4dev_contact_page = """
Contact Us. General Enquiries: help@dox4devuk.com
Address: 128 City Road, London, United Kingdom, EC1V 2NX
Company Details
Dox 4Dev Ltd
Company No. 14085415
VAT No. 441791880
(c) 2025 Dox 4Dev Ltd. Company No. 14085415 | VAT No. 441791880.
"""

pages = {
    "unrealsolutions.com": unreal_solutions_footer_and_body_excerpt,
    "dox4devuk.com":       dox4dev_contact_page,
}

for name, text in pages.items():
    hits = extract(text) or [(None, None)]
    seen = {}
    for nine, ctx in hits:
        if nine is None:
            print(f"{name}: nothing found"); continue
        seen.setdefault(nine, ctx)
    for nine, ctx in seen.items():
        cs = vrn_checksum_valid(nine)
        verdict = "-> ask HMRC" if (ctx and cs) else "dropped"
        print(f"{name}: candidate {nine}, near VAT: {ctx}, checksum ok: {cs} {verdict}")

unrealsolutions.com: nothing found
dox4devuk.com: candidate 441791880, near VAT: True, checksum ok: True -> ask HMRC


`unrealsolutions.com`: correctly nothing, even though the page is full of numbers that could confuse a dumb script (a $299 price, a company number, video specs). None of them sit next to the word "VAT", so nothing fires.

`dox4devuk.com`: one real candidate, `441791880`, sitting right next to "VAT No." twice on the page, and it passes the checksum. This is the one candidate for the whole test.

## 2.4 Trying to verify with HMRC

Tried to actually check `441791880` against HMRC. Hit a wall:
- The old free API: shut down since Feb 2025.
- The new one: needs a registered account, roughly a 2-week wait. Not available this week.
- The human web form: needs a login session and a security token, can't be filled in with a simple request.

I ran `GB441791880` through HMRC's own public checker myself (tax.service.gov.uk/check-vat-number/enter-vat-details). Result: **valid**, registered to Dox 4Dev Ltd. That's the one number in this whole notebook that's gone all the way from candidate to confirmed. Every other HMRC result quoted below (2.7 onward) was checked the same way, by hand, on the same page, not assumed.

## 2.5 What this run actually showed

- 2 real pages tested end to end: one produced no candidate, one produced a candidate that was later confirmed by HMRC (see the update in 2.4). I did not obtain a statistically meaningful false-positive rate from this: with a single candidate produced, the observed false-positive count is 0/1, not a measured production false-positive rate. Calling that "0%" would overstate what a sample of one can tell you.
- 12.5% of companies had a findable website at all
- End to end: 1 out of 16 companies (6.25%) produced a VAT candidate, and that candidate was subsequently confirmed by HMRC. This is one verified result, not a reliable estimate of overall coverage.

What this doesn't tell you: this is one sector, picked to be a hard case, not a national estimate. It probably makes resolution look *easier* than it really is, since search only surfaces companies that are already easy to find online. And the HMRC wall is about *this week specifically*, not forever: get the registered credential and it's gone.

## 2.6 Testing 3 smarter ideas

**Search the web for the pattern first, match to a company after.** Tried broad searches like `"VAT No" "Company No" contact site:co.uk`. Didn't work: results were all "how to find a VAT number" blog posts, not real company pages. My search tool answers questions, it doesn't index raw page text the way a crawler does. Common Crawl itself might do better here, still couldn't test it (same network block as 1.6).

In [7]:
queries_tried = [
    '"VAT No" "Company No" contact site:co.uk -directory',
    '"VAT Reg No" "Company Reg No" online shop UK',
]
outcomes = [
    "10/10 results were advice articles about finding a VAT number, zero real company pages",
    "same pattern: SEO articles dominate, plus one already-known third-party lookup site, no new company",
]
for q, o in zip(queries_tried, outcomes):
    print(f"{q}\n  -> {o}\n")

"VAT No" "Company No" contact site:co.uk -directory
  -> 10/10 results were advice articles about finding a VAT number, zero real company pages

"VAT Reg No" "Company Reg No" online shop UK
  -> same pattern: SEO articles dominate, plus one already-known third-party lookup site, no new company



**Public procurement records.** Found a real trap: government purchase orders often show the *buyer's* VAT number (the department's own), not the supplier's. The same number, `GB287461957`, showed up against 5 completely different supplier names. A script grabbing "VAT number near a company name" would confidently produce 5 wrong matches from one real number.

In [8]:
procurement_hits = [
    {"doc": "PO, UK Shared Business Services", "supplier": "Scaled Agile Inc",           "vat": "GB287461957", "whose": "buyer"},
    {"doc": "PO, STFC",                        "supplier": "10G Networks Ltd",           "vat": "GB287461957", "whose": "buyer"},
    {"doc": "PO, UKRI",                        "supplier": "R-Com Consulting Limited",   "vat": "GB287461957", "whose": "buyer"},
    {"doc": "PO, NERC",                        "supplier": "The Dextrous Web Ltd (dxw)", "vat": "GB287461957", "whose": "buyer"},
    {"doc": "PO, MRC",                         "supplier": "Analytik Jena AG",           "vat": "GB618367325", "whose": "buyer"},
    {"doc": "Invitation to Tender",            "supplier": "(bidder, blank template)",   "vat": "(unfilled field)", "whose": "supplier, but empty"},
]
for h in procurement_hits:
    print(f"{h['doc']:32s} supplier: {h['supplier']:26s} VAT shown: {h['vat']:16s} ({h['whose']})")

PO, UK Shared Business Services  supplier: Scaled Agile Inc           VAT shown: GB287461957      (buyer)
PO, STFC                         supplier: 10G Networks Ltd           VAT shown: GB287461957      (buyer)
PO, UKRI                         supplier: R-Com Consulting Limited   VAT shown: GB287461957      (buyer)
PO, NERC                         supplier: The Dextrous Web Ltd (dxw) VAT shown: GB287461957      (buyer)
PO, MRC                          supplier: Analytik Jena AG           VAT shown: GB618367325      (buyer)
Invitation to Tender             supplier: (bidder, blank template)   VAT shown: (unfilled field) (supplier, but empty)


Also found the fix half-working: tender application forms *do* ask bidders for their VAT number as a required field, exactly as expected. But what gets published on the portal is the blank form, not the winning bidder's filled-in answer.

**Using procurement supplier names as a better starting list.** Picked 2 real suppliers named on real government purchase orders, so already confirmed to be actively trading, unlike a random pick that might be an empty shell.

In [9]:
candidate = "936054523"
total = sum(int(d) * w for d, w in zip(candidate[:7], WEIGHTS))
check = int(candidate[7:9])
scheme = "mod97" if _residual(total) == check else ("mod9755" if _residual(total+55) == check else None)
print(f"GB{candidate}: checksum valid = {scheme is not None} (scheme: {scheme})")

GB936054523: checksum valid = True (scheme: mod97)


- **dxw (The Dextrous Web Ltd):** a third-party site reports `GB936054523`, checksum-valid, computed above. But dxw's *own* site (checked directly) doesn't show it anywhere. So this stays unconfirmed by the primary source, same provenance problem as the third-party lookup sites in 1.7.
- **Volspec:** hit the exact same name-collision problem as 2.2. Two different companies with almost the same name, no VAT found for either.

**Verdict:** procurement seeding buys one real thing (confidence the company is actually trading) and doesn't buy the two things that actually blocked coverage: name collisions, and the fact that VAT disclosure stays voluntary even for a solid, certified, government-vetted supplier.

## 2.7 Expanding the sample

16 active companies was too thin to say much. Wanted more, for two reasons: a bigger sample gives a less noisy website-findable rate, and more candidates means more numbers I can actually take to HMRC's checker myself and confirm, not just claim.

**A limitation worth being upfront about:** this pass couldn't repeat 2.1's exact method. Companies House's advanced search is a form that returns results at a URL with a long query string, and my tooling this time round can only open a URL that a prior search already surfaced, not build one from scratch. So instead of "page 2 of the same advanced search," the extra companies below are real SIC-62020 profiles pulled up by searching for that SIC code plus "incorporated 2022," one search-engine query at a time. Same register, same filter criteria (SIC 62020, incorporated Jan 2022 or later), same honesty rule (logged exactly as found, nothing dropped for being inconvenient), just a different door into it. Flagging this so the method section stays accurate, not because the companies are any less real.

In [ ]:

sample_batch2 = [
    {"name": "DATA ACTIVE MANAGEMENT LIMITED",   "number": "14095664", "status": "Active", "flag": None},
    {"name": "LDTR LTD",                          "number": "14435362", "status": "Active", "flag": None},
    {"name": "ACTIVE INSIGHT LIMITED",             "number": "14237338", "status": "Active", "flag": None},
    {"name": "INTERROBANG LIMITED",                "number": "14466175", "status": "Active", "flag": "trades as Vouchsafe"},
    {"name": "FORFEND INFORMATION SECURITY LTD",   "number": "14029418", "status": "Active", "flag": None},
    {"name": "INFOTRUST COMPANY LIMITED",          "number": "14465667", "status": "Active", "flag": "active proposal to strike off"},
    {"name": "MG & CO CONSULTING LTD",             "number": "14254651", "status": "Active", "flag": None},
    {"name": "INTER-ACTIVE GROUP HOLDINGS LTD",    "number": "14162312", "status": "Active", "flag": "62020 is a secondary SIC, not primary"},
    {"name": "INTERNET EXPERT GROUP LTD",          "number": "14082787", "status": "Active", "flag": None},
    {"name": "OPTIIM LTD",                         "number": "14057986", "status": "Active", "flag": None},
    {"name": "ATTERCOP LTD",                       "number": "14158982", "status": "Active", "flag": None},
    {"name": "WE DO YOUR LTD",                     "number": "14004115", "status": "Active", "flag": None},
    {"name": "CYBERGATE TECHNOLOGIES LTD",         "number": "14222652", "status": "Active", "flag": None},
    {"name": "SHAUN MOORE CONTRACTING LTD",        "number": "13961672", "status": "Active", "flag": None},
    {"name": "MYITGUYUK LTD",                      "number": "14238819", "status": "Active", "flag": "active proposal to strike off"},
    {"name": "AVENSIS IT SERVICES LTD",            "number": "14020867", "status": "Active", "flag": "active proposal to strike off"},
    {"name": "ASPIRE GROUP SERVICES LIMITED",      "number": "14317698", "status": "Active", "flag": "active proposal to strike off"},
    {"name": "LG DIGITAL LTD",                     "number": "14315852", "status": "Active", "flag": "active proposal to strike off"},
    {"name": "DRYAD GLOBAL LTD",                   "number": "14177766", "status": "Active", "flag": "active proposal to strike off"},
]

combined = active + sample_batch2  # 'active' is the 16 from 2.1
flagged = [r for r in sample_batch2 if r["flag"]]
print(f"batch 2: {len(sample_batch2)} more active SIC-62020 companies, {len(flagged)} carry a Companies House risk flag")
print(f"combined sample: {len(combined)} active companies")
for r in sample_batch2:
    tag = f"  [{r['flag']}]" if r["flag"] else ""
    print(" ", r["number"], r["name"] + tag)

batch 2: 19 more active SIC-62020 companies, 8 carry a Companies House risk flag
combined sample: 35 active companies (up from 16)
  14095664 DATA ACTIVE MANAGEMENT LIMITED
  14435362 LDTR LTD
  14237338 ACTIVE INSIGHT LIMITED
  14466175 INTERROBANG LIMITED  [trades as Vouchsafe]
  14029418 FORFEND INFORMATION SECURITY LTD
  14465667 INFOTRUST COMPANY LIMITED  [active proposal to strike off]
  14254651 MG & CO CONSULTING LTD
  14162312 INTER-ACTIVE GROUP HOLDINGS LTD  [62020 is a secondary SIC, not primary]
  14082787 INTERNET EXPERT GROUP LTD
  14057986 OPTIIM LTD
  14158982 ATTERCOP LTD
  14004115 WE DO YOUR LTD
  14222652 CYBERGATE TECHNOLOGIES LTD
  13961672 SHAUN MOORE CONTRACTING LTD
  14238819 MYITGUYUK LTD  [active proposal to strike off]
  14020867 AVENSIS IT SERVICES LTD  [active proposal to strike off]
  14317698 ASPIRE GROUP SERVICES LIMITED  [active proposal to strike off]
  14315852 LG DIGITAL LTD  [active proposal to strike off]
  14177766 DRYAD GLOBAL LTD  [active propo

## 2.8 Running the funnel on the checked slice

Didn't get through all 19 this pass: ran the website step on 6 of them, the same way as 2.2, logging misses as misses instead of skipping them:

- **Data Active Management Ltd (14095664):** real, confirmable website (`activemanagement.org`). No VAT number anywhere on it. Also walked straight into a fresh version of the name-collision trap from 2.2: an unrelated firm called "Active Consulting" (different company number, VAT `GB745242146`) shows up in the same searches. Logged as no-VAT-found, not as a match to the wrong company.
- **Forfend Information Security Ltd (14029418):** real, confirmable website (`forfendinfosec.com`, also independently listed by NCSC and CREST as a real accredited pen-testing firm, about as strong a "this company genuinely trades" signal as this project gets). VAT number itself isn't on Forfend's own site; a third-party company-data site (Endole) reports `GB441604031` for this exact company number. Same provenance shape as dox4dev in 2.3-2.4: passes the checksum (tested below), one third-party source. **Update, checked by hand: run through HMRC's own checker and confirmed valid, registered to Forfend Information Security Ltd.**
- **Interrobang Ltd (14466175):** real, confirmable website, but only findable by its trading name, Vouchsafe (an identity-verification startup), nothing in its own branding says "Interrobang." No VAT number findable under either name. This is 1.2's warning about legal-name-vs-trading-name playing out for real, not just a theoretical risk.
- **MG & Co Consulting Ltd, Optiim Ltd, Cybergate Technologies Ltd:** no findable, confirmable website for any of the three.

Remaining 13 companies from batch 2 haven't been run through the website step yet: queued, not dropped.

In [ ]:


new_candidate = "441604031"
total = sum(int(d) * w for d, w in zip(new_candidate[:7], WEIGHTS))
check = int(new_candidate[7:9])
scheme = "mod97" if _residual(total) == check else ("mod9755" if _residual(total+55) == check else None)
print(f"GB{new_candidate}: checksum valid = {scheme is not None} (scheme: {scheme})")
print("source: Endole (third-party), for Forfend Information Security Ltd, company no. 14029418")
print("HMRC status: checked by hand, confirmed valid")

GB441604031: checksum valid = True (scheme: mod9755)
source: Endole (third-party), for Forfend Information Security Ltd, company no. 14029418
HMRC status: checked by hand, confirmed valid


## 2.9 Where the numbers stand now

| | Part 2 (original, 16) | Part 2.7-2.8 (expanded, 22 checked) |
|---|---|---|
| Companies checked for a website | 16 | 22 |
| Findable, confirmable website | 2 (12.5%) | 5 (22.7%) |
| Checksum-valid VAT candidate produced | 1 | 2 |
| Confirmed by HMRC's own checker, by hand | 0 | 2 (dox4dev, Forfend) |
| Pending manual HMRC check | none | 0 |

The jump from 12.5% to 22.7% on the website-findable rate is **not** a sign the real rate moved: 22 companies is still a small sample, and a single extra hit swings this percentage hard. Treat both numbers as "somewhere in the low-to-mid 20s at most, possibly lower," not as two different facts. 13 companies from batch 2 are still unchecked, so both rows in the second column will move again once those run.

What did *not* change: still one sector (IT consultancy), still picked to be a hard case, still nowhere near a national estimate on its own; see 3.2 for what happens when these rates get projected up to 40,000 suppliers or the whole register.

---

# Part 3: What you'd do with real resources

## 3.1 What changes with real resources

**Keeping the list fresh.** Companies House also has a free real-time feed (Streaming API) for new and closed companies, so the list stays minutes-old instead of month-old.

**Finding websites.** Buy a company-to-domain dataset instead of searching one by one. This probably raises the 12.5% from Part 2, but doesn't fix it: a lot of small companies genuinely have no useful web presence, buying data doesn't invent one.

**Crawling.** Once you know the domain, fetch only a few pages per company (terms, contact, homepage), not the whole web, so it's cheap. Some sites will block a crawler; this notebook's own experience (the 50-request block in 1.7) is a preview of that at bigger scale, so budget for proxies.

**Procurement and council records.** No shortcut here. Every source has its own file format, that's people-hours, not something a bigger server fixes.

**Verifying.** The one part that gets genuinely solved with resources, not just sped up.

Even the whole country's VAT register clears in about 8 days of nonstop querying, at HMRC's own documented rate. Once you have the credential, verifying is basically free and fast. **It was never really the hard part.**

One thing I checked that didn't pay off: since March 2024, every UK company must give Companies House a registered email address. Looked like a free source of company domains for a moment. It isn't: Companies House keeps that address private, for their own use only, never on the public record.

## 3.2 What it would cost, and how far that gets you

Putting a number on this, built from what's actually measured in this notebook plus two real, sourced market figures (not invented):

**The two real inputs:**
- Buying a company→domain lookup instead of searching one by one (the fix proposed in 3.1): company-record enrichment APIs (People Data Labs, Explorium, and similar) publish **roughly \$0.01-\$0.10 per company record** at self-serve volume, source-checked against current vendor pricing pages.
- HMRC's own checker, once you have the registered credential, is **free** and rate-limited to roughly 3 requests/second (this is also where 3.1's "8 days to clear the whole 2.18M register" comes from: 2,180,000 ÷ 3 ≈ 8.4 days of continuous querying). Verification cost is effectively zero next to the domain-lookup step.

**For the customer's actual 40,000 suppliers:**

| Step | Cost | Basis |
|---|---|---|
| Domain lookup, all 40,000 | \$400 - \$4,000 (one-off) | market rate above × 40,000 |
| HMRC checks on whatever candidates come out | ~free, ~a few hours of API time | 3 req/sec, no per-call charge |
| **Blended cost per supplier attempted** | **≈\$0.01-\$0.10** | dominated entirely by the domain-lookup step |

That's cost per company *attempted*, not per VAT number *found*, and those are very different numbers here, because most attempts still return nothing.

**Coverage, on this notebook's own measured rates (2.9):** of the 22 companies actually pushed through the funnel, 22.7% had a findable website and 9.1% ended up producing a checksum-valid candidate at all. Applying that 9.1% straight to 40,000 suppliers gives **≈3,640 candidate VAT numbers**, which puts cost per *verified* number at roughly **\$0.11-\$1.10**, still cheap, but the honest headline is the 9.1%, not the dollar figure: *roughly 9 in 10 suppliers on this sector's numbers would come back with nothing at all*, no matter how much gets spent on the lookup step.

**Why this is a floor, not a forecast:**
- IT consultancy was picked on purpose in 2.1 because it was expected to score badly (no online checkout, no legal disclosure duty). A customer's real 40,000 suppliers will span every sector, including e-commerce, where source 1 in 1.3's table is a legal requirement, not a maybe. National coverage should sit above 9.1%, potentially well above it, but this notebook has no measured number for anything except IT consultancy.
- 22 companies is small enough that this whole estimate should be read as "same order of magnitude," not "accurate to the percentage point." The gap between 12.5% and 22.7% across one extra batch of 6 companies is the visible proof of that.


## 3.3 What breaks first, what to watch

1. **Finding a website stays the ceiling.** Buying data helps, doesn't remove it.
2. **Bot-blocking on the crawl.** Same shape as the 50-request block already hit, worse at scale.
3. **Every new records source needs its own one-off setup.** Council file, procurement portal, doesn't parallelise with money.
4. **Old crawled pages go stale.** Footers change, VAT numbers get cancelled.
5. **Verifying does *not* break**, per 3.1's own math. The one stage real resources genuinely fix rather than just speed up.

Worth watching in production: candidate yield per source, HMRC match rate, how close name-matches are drifting to the cutoff, dead links on already-found domains, how far behind the Streaming API feed gets, review-queue backlog, and cost per verified record, the number that ties it all together.